In [ ]:
# Load the details of the currently known farm from a JSON

import json
from typing import List, Dict, Any
from pydantic import BaseModel

class Crop(BaseModel):
    name: str

class Activities(BaseModel):
    plowing: str = None
    discing: str = None
    sowing: str = None
    herbicide: str = None
    first_cut: str = None
    second_cut: str = None

class Location(BaseModel):
    type: str
    coordinates: List[List[List[float]]]

class Field(BaseModel):
    crop: Crop
    activities: Activities
    location: Location

class Farm(BaseModel):
    name: str
    fields: List[Field]

class FarmData(BaseModel):
    farm: Farm

# Read the JSON file
with open("../local_data/cluj_farm_1.json", "r") as f:
    data = json.load(f)

# Parse into the data model
farm_data = FarmData(**data)
print(farm_data)

In [ ]:
# Simulated daily temperatures (°C)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

daily_temperatures_simulation = {
    "date": pd.date_range("2025-04-01", "2025-09-30", freq="D"),
    "tmin": 10 + 5 * np.sin(np.linspace(0, 3.14, 183)),  # synthetic temp pattern
    "tmax": 20 + 8 * np.sin(np.linspace(0, 3.14, 183)),
}
df = pd.DataFrame(daily_temperatures_simulation)

df.tail(10)


In [ ]:
# Compute daily GDD for Lucerne

base_temp = 5  # °C - base temperature for Lucerne
df["tavg"] = (df["tmin"] + df["tmax"]) / 2
df["gdd"] = (df["tavg"] - base_temp).clip(lower=0)  # no negative accumulation
df["gdd_cum"] = df["gdd"].cumsum()

df.tail(10)

In [ ]:
# Map GDD thresholds to BBCH stages (example for Lucerne)

bbch_stages_lucerne = [
    (0, 150, "00–09: Germination"),
    (150, 300, "10–19: Leaf development"),
    (300, 600, "30–39: Stem elongation"),
    (600, 900, "50–59: Flower buds"),
    (900, 1200, "60–69: Flowering"),
    (1200, 1500, "70–79: Fruit development"),
    (1500, 9999, "80–89: Ripening / senescence"),
]

def get_bbch(gdd):
    for low, high, stage in bbch_stages_lucerne:
        if low <= gdd < high:
            return stage
    return "Unknown"

df["BBCH_stage"] = df["gdd_cum"].apply(get_bbch)

df.tail(10)

In [ ]:
# Plot cumulative GDD for Lucerne

plt.figure(figsize=(12, 6))
plt.plot(df["date"], df["gdd_cum"], color="forestgreen", linewidth=2)
plt.xlabel("Date")
plt.ylabel("Cumulative GDD (°C·days)")
plt.title("Lucerne – Cumulative GDD and BBCH Stages")
plt.grid(True)

# detect transitions
last_stage = None
for i, row in df.iterrows():
    if row["BBCH_stage"] != last_stage:
        plt.axvline(x=row["date"], color="gray", linestyle="--", alpha=0.4)
        plt.text(row["date"], row["gdd_cum"] + 30, row["BBCH_stage"],
                 rotation=45, fontsize=9, color="darkslategray")
        last_stage = row["BBCH_stage"]

plt.tight_layout()
plt.show()

In [ ]:
# Sample code for downloading data from CDS

import cdsapi

dataset = "reanalysis-era5-land"
request = {
    "variable": [
        # Temperature to which the air, at 2 metres above the surface of the Earth, 
        # would have to be cooled for saturation to occur.It is a measure of the 
        # humidity of the air. Combined with temperature and pressure, it can be 
        # used to calculate the relative humidity. 2m dew point temperature is 
        # calculated by interpolating between the lowest model level and the Earth's 
        # surface, taking account of the atmospheric conditions. Temperature measured 
        # in kelvin can be converted to degrees Celsius (°C) by subtracting 273.15.
        "2m_dewpoint_temperature",

        # Temperature of air at 2m above the surface of land, sea or in-land waters. 
        # 2m temperature is calculated by interpolating between the lowest model level
        # and the Earth's surface, taking account of the atmospheric conditions. 
        # Temperature measured in kelvin can be converted to degrees Celsius (°C) by 
        # subtracting 273.15.
        "2m_temperature",

        # Temperature of the surface of the Earth. The skin temperature is the 
        # theoretical temperature that is required to satisfy the surface energy 
        # balance. It represents the temperature of the uppermost surface layer, 
        # which has no heat capacity and so can respond instantaneously to changes 
        # in surface fluxes. Skin temperature is calculated differently over land 
        # and sea. Temperature measured in kelvin can be converted to degrees 
        # Celsius (°C) by subtracting 273.15.
        "skin_temperature",

        # Temperature of the soil in layer 1 (0 - 7 cm) of the ECMWF Integrated Forecasting 
        # System. The surface is at 0 cm. Soil temperature is set at the middle of each 
        # layer, and heat transfer is calculated at the interfaces between them. It is 
        # assumed that there is no heat transfer out of the bottom of the lowest layer. 
        # Temperature measured in kelvin can be converted to degrees Celsius (°C) by 
        # subtracting 273.15.
        "soil_temperature_level_1",

        # Temperature of the soil in layer 2 (7 -28cm) of the ECMWF Integrated Forecasting System.
        "soil_temperature_level_2",

        # Temperature of the soil in layer 3 (28-100cm) of the ECMWF Integrated Forecasting System.
        "soil_temperature_level_3",

        # Temperature of the soil in layer 4 (100-289 cm) of the ECMWF Integrated Forecasting System.
        "soil_temperature_level_4",

        # Amount of water in the vegetation canopy and/or in a thin layer on the soil. 
        # It represents the amount of rain intercepted by foliage, and water from dew. 
        # The maximum amount of 'skin reservoir content' a grid box can hold depends 
        # on the type of vegetation, and may be zero. Water leaves the 'skin reservoir' 
        # by evaporation.
        "skin_reservoir_content",

        # Volume of water in soil layer 1 (0 - 7 cm) of the ECMWF Integrated Forecasting 
        # System. The surface is at 0 cm. The volumetric soil water is associated with 
        # the soil texture (or classification), soil depth, and the underlying groundwater level.
        "volumetric_soil_water_layer_1",
        
        # Volume of water in soil layer 2 (7 -28 cm) of the ECMWF Integrated Forecasting System.
        "volumetric_soil_water_layer_2",

        # Volume of water in soil layer 3 (28-100 cm) of the ECMWF Integrated Forecasting System. 
        "volumetric_soil_water_layer_3",

        # Volume of water in soil layer 4 (100-289 cm) of the ECMWF Integrated Forecasting System.
        "volumetric_soil_water_layer_4",

        # The amount of evaporation from bare soil at the top of the land surface. This 
        # variable is accumulated from the beginning of the forecast time to the end of the forecast step.
        "evaporation_from_bare_soil",

        # Amount of evaporation from surface water storage like lakes and 
        # inundated areas but excluding oceans. This variable is accumulated 
        # from the beginning of the forecast time to the end of the forecast step.
        "evaporation_from_open_water_surfaces_excluding_oceans",

        # The amount of evaporation from the canopy interception reservoir at 
        # the top of the canopy. This variable is accumulated from the beginning 
        # of the forecast time to the end of the forecast step.
        "evaporation_from_the_top_of_canopy",

        # Amount of evaporation from vegetation transpiration. This has the 
        # same meaning as root extraction i.e. the amount of water extracted 
        # from the different soil layers. This variable is accumulated from 
        # the beginning of the forecast time to the end of the forecast step.
        "evaporation_from_vegetation_transpiration",

        # Potential evaporation (pev) in the current ECMWF model is computed, 
        # by making a second call to the surface energy balance routine with 
        # the vegetation variables set to "crops/mixed farming" and assuming 
        # no stress from soil moisture. In other words, evaporation is computed 
        # for agricultural land as if it is well watered and assuming that the
        # atmosphere is not affected by this artificial surface condition. 
        # The latter may not always be realistic. Although pev is meant to 
        # provide an estimate of irrigation requirements, the method can 
        # give unrealistic results in arid conditions due to too strong 
        # evaporation forced by dry air. Note that in ERA5-Land pev is computed 
        # as an open water evaporation (Pan evaporation) and assuming that 
        # the atmosphere is not affected by this artificial surface condition. 
        # The latter is different from the way pev is computed in ERA5. 
        # This variable is accumulated from the beginning of the forecast time 
        # to the end of the forecast step.
        "potential_evaporation",

        # Some water from rainfall, melting snow, or deep in the soil, stays
        # stored in the soil. Otherwise, the water drains away, either over 
        # the surface (surface runoff), or under the ground (sub-surface 
        # runoff) and the sum of these two is simply called 'runoff'. This 
        # variable is the total amount of water accumulated from the beginning 
        # of the forecast time to the end of the forecast step. The units of 
        # runoff are depth in metres. This is the depth the water would have i
        #f it were spread evenly over the grid box. Care should be taken when 
        # comparing model variables with observations, because observations are 
        # often local to a particular point rather than averaged over a grid 
        # square area. Observations are also often taken in different units, 
        # such as mm/day, rather than the accumulated metres produced here. 
        # Runoff is a measure of the availability of water in the soil, and 
        # can, for example, be used as an indicator of drought or flood. 
        # More information about how runoff is calculated is given in the
        # IFS Physical Processes documentation.
        "runoff",

        # Evaporation from snow averaged over the grid box (to find flux over
        # snow, divide by snow fraction). This variable is accumulated from
        # the beginning of the forecast time to the end of the forecast step.
        "snow_evaporation",

        # Some water from rainfall, melting snow, or deep in the soil, stays
        # stored in the soil. Otherwise, the water drains away, either over 
        # the surface (surface runoff), or under the ground (sub-surface runoff) 
        # and the sum of these two is simply called 'runoff'. This variable is 
        # accumulated from the beginning of the forecast time to the end of the 
        # forecast step. The units of runoff are depth in metres. This is the 
        # depth the water would have if it were spread evenly over the grid box. 
        # Care should be taken when comparing model variables with observations, 
        # because observations are often local to a particular point rather than 
        # averaged over a grid square area. Observations are also often taken 
        # in different units, such as mm/day, rather than the accumulated metres
        # produced here. Runoff is a measure of the availability of water in the
        # soil, and can, for example, be used as an indicator of drought or flood.
        # More information about how runoff is calculated is given in the IFS 
        # Physical Processes documentation.
        "sub_surface_runoff",

        # Some water from rainfall, melting snow, or deep in the soil, stays 
        # stored in the soil. Otherwise, the water drains away, either over 
        # the surface (surface runoff), or under the ground (sub-surface runoff) 
        # and the sum of these two is simply called 'runoff'. This variable is 
        # the total amount of water accumulated from the beginning of the forecast
        # time to the end of the forecast step. The units of runoff are depth in
        # metres. This is the depth the water would have if it were spread evenly
        # over the grid box. Care should be taken when comparing model variables
        # with observations, because observations are often local to a particular
        # point rather than averaged over a grid square area. Observations are
        # also often taken in different units, such as mm/day, rather than the
        # accumulated metres produced here. Runoff is a measure of the availability
        # of water in the soil, and can, for example, be used as an indicator of
        # drought or flood. More information about how runoff is calculated is
        # given in the IFS Physical Processes documentation.
        "surface_runoff",

        # Accumulated amount of water that has evaporated from the Earth's surface,
        # including a simplified representation of transpiration (from vegetation),
        # into vapour in the air above. This variable is accumulated from the
        # beginning of the forecast to the end of the forecast step. The ECMWF 
        # Integrated Forecasting System convention is that downward fluxes are 
        # positive. Therefore, negative values indicate evaporation and positive
        # values indicate condensation.
        "total_evaporation",

        # Eastward component of the 10m wind. It is the horizontal speed of
        # air moving towards the east, at a height of ten metres above the
        # surface of the Earth, in metres per second. Care should be taken
        # when comparing this variable with observations, because wind observations
        # vary on small space and time scales and are affected by the local terrain,
        # vegetation and buildings that are represented only on average in the ECMWF
        # Integrated Forecasting System. This variable can be combined with the V
        # component of 10m wind to give the speed and direction of the horizontal 10m wind.
        "10m_u_component_of_wind",

        # Northward component of the 10m wind. It is the horizontal speed of air
        # moving towards the north, at a height of ten metres above the surface 
        # of the Earth, in metres per second. Care should be taken when comparing
        # this variable with observations, because wind observations vary on small
        # space and time scales and are affected by the local terrain, vegetation
        # and buildings that are represented only on average in the ECMWF Integrated
        # Forecasting System. This variable can be combined with the U component of
        # 10m wind to give the speed and direction of the horizontal 10m wind.
        "10m_v_component_of_wind",
        
        # Pressure (force per unit area) of the atmosphere on the surface of land,
        # sea and in-land water. It is a measure of the weight of all the air in a
        # column vertically above the area of the Earth's surface represented at a
        # fixed point. Surface pressure is often used in combination with temperature
        # to calculate air density. The strong variation of pressure with altitude
        # makes it difficult to see the low and high pressure systems over mountainous
        # areas, so mean sea level pressure, rather than surface pressure, is normally
        # used for this purpose. The units of this variable are Pascals (Pa). Surface 
        # pressure is often measured in hPa and sometimes is presented in the old units
        # of millibars, mb (1 hPa = 1 mb = 100 Pa).
        "surface_pressure",

        # Accumulated liquid and frozen water, including rain and snow, that falls 
        # to the Earth's surface. It is the sum of large-scale precipitation (that 
        # precipitation which is generated by large-scale weather patterns, such as
        # troughs and cold fronts) and convective precipitation (generated by convection
        # which occurs when air at lower levels in the atmosphere is warmer and less
        # dense than the air above, so it rises). Precipitation variables do not
        # include fog, dew or the precipitation that evaporates in the atmosphere 
        # before it lands at the surface of the Earth. This variable is accumulated
        # from the beginning of the forecast time to the end of the forecast step.
        # The units of precipitation are depth in metres. It is the depth the water
        # would have if it were spread evenly over the grid box. Care should be taken
        # when comparing model variables with observations, because observations are 
        # often local to a particular point in space and time, rather than representing
        # averages over a model grid box and model time step.
        "total_precipitation",

        # One-half of the total green leaf area per unit horizontal ground surface 
        # area for high vegetation type.
        "leaf_area_index_high_vegetation",

        # One-half of the total green leaf area per unit horizontal ground surface
        # area for low vegetation type.
        "leaf_area_index_low_vegetation",

        "high_vegetation_cover",
        "glacier_mask",
        "lake_cover",
        "low_vegetation_cover",
        "lake_total_depth",
        "land_sea_mask",
        "soil_type",
        "type_of_high_vegetation",
        "type_of_low_vegetation"
    ],
    "year": "2025",
    "month": "09",
    "day": [
        "01", "03",
        "04", "06",
        "07", "09",
        "10", "12",
        "13", "15",
        "16", "18",
        "19", "21",
        "22", "24",
        "25", "27",
        "28"
    ],
    "time": [
        "08:00",
        "14:00",
        "21:00"
    ],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [46.9, 24, 46.7, 24.2]
}

client = cdsapi.Client()
client.retrieve(dataset, request, "../local_data/temp_septembrie.grib")


In [1]:
import xarray as xr

temperature_file = "../local_data/temp_aprilie.grib"

ds = xr.open_dataset(temperature_file)
print(ds)
df = ds.to_dataframe().reset_index()
print(df.head())

skipping corrupted Message
Traceback (most recent call last):
  File "/Users/bogdanmel/miniconda3/envs/tf/lib/python3.11/site-packages/cfgrib/messages.py", line 274, in itervalues
    yield self.filestream.message_from_file(file, errors=errors)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bogdanmel/miniconda3/envs/tf/lib/python3.11/site-packages/cfgrib/messages.py", line 341, in message_from_file
    return Message.from_file(file, offset, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bogdanmel/miniconda3/envs/tf/lib/python3.11/site-packages/cfgrib/messages.py", line 97, in from_file
    codes_id = eccodes.codes_grib_new_from_file(file)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bogdanmel/miniconda3/envs/tf/lib/python3.11/site-packages/gribapi/gribapi.py", line 413, in grib_new_from_file
    GRIB_CHECK(err)
  File "/Users/bogdanmel/miniconda3/envs/tf/lib/python3.11/site-packages/gribapi/gribapi.

EOFError: No valid message found: '../local_data/temp_aprilie.grib'

In [ ]:
# Required libs
import xarray as xr
import rioxarray          # optional, for raster/vector handling
import geopandas as gpd
import shapely.geometry as geom
import numpy as np
import pandas as pd

# -------------------------
# User config
# -------------------------
file_path = "era5_land_daily.nc"   # or .grib ; use engine cfgrib if needed
polygon_lonlat = [                 # example polygon (lon, lat)
    (24.17629055822008, 46.83272043023766),
    (24.176730440479606, 46.83318284358206),
    (24.177159593903536, 46.83320486316589),
    (24.177878425888625, 46.83290392807272),
    (24.17746000130029, 46.832368112638086),
]
tbase = 5.0  # base temp for lucerne; change per crop

# -------------------------
# Open dataset (try netCDF first, fallback to cfgrib)
# -------------------------
try:
    ds = xr.open_dataset(file_path)   # NetCDF
except Exception as e:
    # if GRIB use cfgrib engine
    ds = xr.open_dataset(file_path, engine="cfgrib")

print(ds)

# -------------------------
# Make polygon mask (geopandas) and compute spatial mean
# -------------------------
poly = geom.Polygon(polygon_lonlat)
gdf = gpd.GeoDataFrame({"geometry":[poly]}, crs="EPSG:4326")

# Reproject dataset coordinates to lon/lat order used by xarray (should be lon,lat)
# We'll do a simple spatial averaging: select grid points inside polygon
# Function: get spatial mean time series for a given dataarray
def spatial_mean_over_polygon(da, polygon):
    # da: xarray DataArray with dims (time, latitude, longitude) or (time, lat, lon)
    # polygon: shapely polygon in lon/lat
    lons = da.coords[da.dims[-1]].values
    lats = da.coords[da.dims[-2]].values
    # create 2D lon/lat mesh
    lon2d, lat2d = np.meshgrid(lons, lats)
    # flatten and test points inside polygon
    points = np.vstack([lon2d.ravel(), lat2d.ravel()]).T
    import shapely.vectorized as sv
    mask_flat = sv.contains(polygon, points[:,0], points[:,1])
    mask = mask_flat.reshape(lat2d.shape)
    # convert mask to DataArray with same dims (lat,lon)
    mask_da = xr.DataArray(mask, coords={da.dims[-2]: lats, da.dims[-1]: lons}, dims=(da.dims[-2], da.dims[-1]))
    # multiply and compute mean over spatial dims (ignore NaN)
    weighted = da.where(mask_da)
    return weighted.mean(dim=[da.dims[-2], da.dims[-1]])

# -------------------------
# Helper: get variable if exists with alternative names
# -------------------------
def get_var(ds, candidates):
    for v in candidates:
        if v in ds:
            return ds[v]
    raise KeyError(f"None of {candidates} found in dataset. Available: {list(ds.data_vars)}")

# Map variables (adjust names to your dataset)
dew = get_var(ds, ["2m_dewpoint_temperature","d2m","dewpoint_temperature"])
t2m = get_var(ds, ["2m_temperature","t2m","2t"])
skin_temp = get_var(ds, ["skin_temperature","skin"])
soil_t1 = get_var(ds, ["soil_temperature_level_1","stl1"])
soil_t2 = get_var(ds, ["soil_temperature_level_2","stl2"])
soil_t3 = get_var(ds, ["soil_temperature_level_3","stl3"])
soil_t4 = get_var(ds, ["soil_temperature_level_4","stl4"])
skin_res = get_var(ds, ["skin_reservoir_content","sr"])
vsw1 = get_var(ds, ["volumetric_soil_water_layer_1","swvl1"])
vsw2 = get_var(ds, ["volumetric_soil_water_layer_2","swvl2"])
vsw3 = get_var(ds, ["volumetric_soil_water_layer_3","swvl3"])
vsw4 = get_var(ds, ["volumetric_soil_water_layer_4","swvl4"])
albedo = get_var(ds, ["forecast_albedo","albedo"])
lat_flux = get_var(ds, ["surface_latent_heat_flux","slhf"])
net_solar = get_var(ds, ["surface_net_solar_radiation","ssr"])
net_thermal = get_var(ds, ["surface_net_thermal_radiation","str"])
sens_flux = get_var(ds, ["surface_sensible_heat_flux","sshf"])
swd = get_var(ds, ["surface_solar_radiation_downwards","ssrd"])
twd = get_var(ds, ["surface_thermal_radiation_downwards","strd"])
u10 = get_var(ds, ["10m_u_component_of_wind","u10"])
v10 = get_var(ds, ["10m_v_component_of_wind","v10"])
sp = get_var(ds, ["surface_pressure","sp"])
tp = get_var(ds, ["total_precipitation","tp"])
lai_high = get_var(ds, ["leaf_area_index_high_vegetation","lai_h"])
lai_low = get_var(ds, ["leaf_area_index_low_vegetation","lai_l"])

# -------------------------
# Compute spatial means time series for key fields
# -------------------------
vars_to_avg = {
    "t2m": t2m, "dew": dew, "skin_temp": skin_temp,
    "soil_t1": soil_t1, "soil_t2": soil_t2, "soil_t3": soil_t3, "soil_t4": soil_t4,
    "vsw1": vsw1, "vsw2": vsw2, "vsw3": vsw3, "vsw4": vsw4,
    "lat_flux": lat_flux, "net_solar": net_solar, "swd": swd,
    "u10": u10, "v10": v10, "tp": tp, "albedo": albedo,
    "sp": sp, "lai_h": lai_high, "lai_l": lai_low
}
series = {}
for name, da in vars_to_avg.items():
    try:
        s = spatial_mean_over_polygon(da, poly)
        # ensure time dimension present -> convert to pandas Series
        s = s.to_series()
        series[name] = s
    except Exception as e:
        print(f"Warning: failed spatial mean for {name}: {e}")

# Combine into DataFrame
df = pd.DataFrame(series)
df.index.name = "time"
# convert units as needed (ERA5 often in K/ m / W/m2 / m3 m-3 / meters)
# temperature K->C
if (df["t2m"].mean() > 200):
    df["t2m_C"] = df["t2m"] - 273.15
else:
    df["t2m_C"] = df["t2m"]

if "dew" in df.columns:
    df["dew_C"] = df["dew"] - 273.15 if df["dew"].mean() > 200 else df["dew"]

# precipitation often in m -> mm
if "tp" in df.columns:
    # some datasets give cumulative totals; check units
    df["tp_mm"] = df["tp"] * 1000.0

# wind speed
df["wind_speed"] = np.sqrt(df["u10"]**2 + df["v10"]**2)

# daily Tmin/Tmax/Tavg (if dataset is hourly or daily aggregated adjust accordingly)
# If data is already daily aggregated, t2m_C likely daily mean. But to be safe:
# resample daily if time resolution subdaily
if df.index.freq is None:
    # try resample by day
    df_daily = df.resample("D").agg({
        "t2m_C": ["min","max","mean"],
        "tp_mm": "sum",
        "wind_speed": "mean",
        "lat_flux": "mean",
        "swd": "mean",
        "vsw1":"mean","vsw2":"mean","vsw3":"mean","vsw4":"mean",
        "lai_h":"mean","lai_l":"mean","albedo":"mean"
    })
    # flatten columns
    df_daily.columns = ["_".join(col).strip() for col in df_daily.columns.values]
else:
    df_daily = df  # assume already daily

# Rename sensible columns
df_daily = df_daily.rename(columns={
    "t2m_C_min":"tmin_C", "t2m_C_max":"tmax_C", "t2m_C_mean":"tavg_C",
    "tp_mm_sum":"precip_mm", "wind_speed_mean":"wind_speed",
    "lat_flux_mean":"lat_flux", "swd_mean":"swd",
    "vsw1_mean":"vsw1","vsw2_mean":"vsw2","vsw3_mean":"vsw3","vsw4_mean":"vsw4",
    "lai_h_mean":"lai_h","lai_l_mean":"lai_l","albedo_mean":"albedo"
})

# -------------------------
# Derived agronomic indicators
# -------------------------
# GDD (base tbase)
df_daily["gdd_daily"] = np.maximum(0, df_daily["tavg_C"] - tbase)
df_daily["gdd_cum"] = df_daily["gdd_daily"].cumsum()

# VPD calculation (approx; from Tetens formula)
# saturation vapour pressure (hPa)
def es(T):
    return 0.6108 * np.exp((17.27 * T) / (T + 237.3))
if "dew_C" in df.columns:
    df_daily["vpd_kPa"] = es(df_daily["tavg_C"]) - es(df_daily["dew_C"])
else:
    # approximate dew = tmin
    df_daily["vpd_kPa"] = es(df_daily["tavg_C"]) - es(df_daily["tmin_C"])

# ET estimate from latent heat flux (W/m2 -> mm/day)
# latent flux in W/m2; convert: mm/day = (W/m2 * 86400) / latent_heat_of_vaporization(J/kg)
L_v = 2.45e6  # J/kg
if "lat_flux" in df_daily.columns:
    df_daily["ET_mm"] = df_daily["lat_flux"] * 86400.0 / L_v
else:
    df_daily["ET_mm"] = np.nan

# Soil water: compute root-zone average (simple weighted sum)
# weights assume layer thickness: l1=0-7cm, l2=7-28cm, l3=28-100cm, l4=100-289cm (example)
layer_depths = np.array([0.07, 0.21, 0.72, 1.89])  # m
if all(c in df_daily.columns for c in ["vsw1","vsw2","vsw3","vsw4"]):
    df_daily["rootzone_vwc"] = (df_daily["vsw1"]*layer_depths[0] + df_daily["vsw2"]*layer_depths[1] +
                                df_daily["vsw3"]*layer_depths[2] + df_daily["vsw4"]*layer_depths[3]) / layer_depths.sum()
    # normalize to 0-1 using min/max observed as proxy for WP-FC unknown
    vmin, vmax = df_daily["rootzone_vwc"].quantile(0.05), df_daily["rootzone_vwc"].quantile(0.95)
    df_daily["rootzone_vwc_norm"] = ((df_daily["rootzone_vwc"] - vmin) / (vmax - vmin)).clip(0,1)
else:
    df_daily["rootzone_vwc"] = np.nan
    df_daily["rootzone_vwc_norm"] = np.nan

# Crop Water Stress Index (simple): 1 - normalized soil water
df_daily["CWSI_simple"] = 1.0 - df_daily["rootzone_vwc_norm"]

# LAI combined
if "lai_h" in df_daily.columns and "lai_l" in df_daily.columns:
    df_daily["LAI_total"] = df_daily["lai_h"].fillna(0) + df_daily["lai_l"].fillna(0)

# Save to CSV
df_daily.to_csv("farm_derived_daily.csv")
print("Saved farm_derived_daily.csv")

# Show sample
df_daily[["tmin_C","tmax_C","tavg_C","gdd_daily","gdd_cum","precip_mm","ET_mm","rootzone_vwc","rootzone_vwc_norm","CWSI_simple","vpd_kPa","LAI_total"]].head(15)
